In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-5: PPO in depth

## Where this session fits

> **The limitation we start from: C1 showed one PPO step in outline. Running PPO on a language model takes four networks, a schedule and a critic that must keep up.**

- **The question:** what does PPO need in practice, and where does it go wrong?
- **Position: session 5 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-4 used GRPO because its reward was a program
    - this session opens the PPO machinery that RLHF (RLLM-2) and many reasoning recipes build on
    - the next session, RLLM-6, is GRPO and its refinements
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. name the models in a PPO run on a language model, and say which are updated, and when
2. compute per-token rewards, TD errors and GAE advantages for a short completion
3. say what the critic buys, and what it costs on a language model
4. read the signals of a PPO run (reward, KL, value loss, clip fraction, entropy) and say what each warns of
5. compare PPO and GRPO on advantage, networks and credit

**What we do not repeat:** the TD error, GAE and the clipped objective are B3 and B4. We use them.


# The models in a PPO run

**Why this section?** The first source of confusion is how many networks there are, and which ones learn.

> **Four networks matter: the policy and the critic are updated; the reference model and the reward model are frozen.**

| Model | Input and output | Updated? | Role |
|:--|:--|:--|:--|
| **policy** $\pi_\theta$ | prefix → distribution over the next token | every mini-batch | the model we are training: samples the completions |
| **critic** $\statevalfun_\phi$ | prefix → one number | every mini-batch | estimates the return from this prefix: the baseline |
| **reference** $\pi_{\text{ref}}$ | prefix → distribution over the next token | never | the model before RL: the anchor for the KL reward |
| **reward model** | prompt and completion → one number | never (in RL) | the score at the last token (RLLM-2); a program, if the reward is verifiable |

**Three more things to place**

- the **old policy** $\pi_{\text{old}}$ (B4) is the policy that generated the batch
    - it is a snapshot, not a fifth network to train
    - many implementations store its log-probabilities when sampling, and need no second copy of the weights
- the policy and the reference start as copies of the supervised fine-tuned model (C1)
    - the critic is often the same model with a new output layer, or starts from the reward model
    - the reward model was trained separately (RLLM-2)
- the **three KLs** of C1 now have their places
    - old versus new policy: the clip, inside the loss
    - policy versus reference: the KL reward, inside the rewards
    - GRPO's loss term: RLLM-6


## One iteration, in order

**Why this section?** The schedule explains why some models are fresh and some are stale.

> **PPO alternates a costly generation phase with several cheap update epochs on the same batch.**

<img src="images/rllm5_loop.png" width=80%>

- **rollout:** generate completions from the current policy
    - usually the main cost, so the batch is reused
- **rewards:** one score per completion, and a KL term for every token (C1)
- **advantages:** the critic values every prefix, and GAE (B3) turns values and rewards into advantages
- **$K$ epochs:** the same batch, cut into mini-batches
    - each mini-batch updates the policy (clipped, B4) and the critic
- **refresh:** the updated policy becomes the sampler for the next batch

**Why the batch goes stale, and why clipping the policy also protects the critic**

- the batch was sampled by $\pi_{\text{old}}$, but after the first mini-batch the policy is no longer $\pi_{\text{old}}$
- supervised learning has fixed labels; here the data is the model's own earlier behavior
- two things go stale
    - the **samples**: they come from a distribution the policy has left (the ratio and the clip correct for this, B4)
    - the **critic**: it was fit to the policy of the last iteration
- keeping the policy near $\pi_{\text{old}}$ limits both
    - the advantages stay meaningful, because the critic is still roughly right
    - the importance ratio stays small, so the correction stays accurate


## The per-token reward

**Why this section?** In RLHF-style PPO, every token gets a reward, though the score arrives only at the end.

> **The reward at token $\tt$ is the KL term $-\beta \log(\pi_\theta / \pi_{\text{ref}})$, plus the score if $\tt$ is the last token.**

- this is C1's per-token reward
- the critic and GAE then treat it like any reward sequence
- the KL term is **sampled**: the log-ratio of the token that was drawn, not the full KL over the vocabulary
    - summed over a completion it estimates $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$
    - a single token's term can be positive or negative
- $\beta$ sets the leash
    - large: the policy stays near the reference, and the score has less influence
    - small: the score dominates, and the proxy is easier to exploit
- notation: the PPO ratio is $r_\tt(\theta)$, as in B4 and the PPO paper
    - the reward is $\rew_\tt$, the same $\rew$ as in the rest of the course
    - the argument $(\theta)$ and the context tell them apart


## Recall: one score becomes a weight for each token

**Why this section?** The critic's job, from C1, is the base for the rest of the session. Here is the sparse case again.

> **A token is judged by what it changed, not by the final score alone: a good start in a bad completion is blamed less than the tokens after it.**

<img src="images/c1_credit.png" width=75%>

- surprise $\delta_\tt$ (the TD error of B3): the reward, plus the next prefix's value, minus this prefix's value
- the advantage adds a token's surprise and all later ones: GAE with $\lambda = 1$
- C1's example had no KL term and $\disc = 1$
- **this session adds two things:** a small reward at every token, and a $\lambda$ below $1$


## Worked example: credit for each token

**Why this section?** The same example, now with the KL reward at every token and three settings of $\lambda$.

> **With a critic, each token gets its own advantage. The dial $\lambda$ sets how much of the end each token feels: a good early token is praised only at a small $\lambda$.**

- a four-token completion, with $\gamma = 1$ and $\beta = 0.1$
- the value after the last token is $0$: the completion has ended
| | token 1 | token 2 | token 3 | token 4 |
|:--|:--|:--|:--|:--|
| log-ratio $\log \pi_\theta / \pi_{\text{ref}}$ | 1.0 | 0.0 | 0.5 | 1.0 |
| KL reward $-\beta \times$ log-ratio | -0.10 | -0.00 | -0.05 | -0.10 |
| score at the last token | 0 | 0 | 0 | +0.20 |
| reward $\rew_\tt$ | -0.10 | +0.00 | -0.05 | +0.10 |
| critic value $\statevalfun_\phi$ of the prefix before the token | 0.50 | 0.90 | 0.90 | 0.90 |
| TD error $\delta_\tt$ | +0.30 | +0.00 | -0.05 | -0.80 |
| advantage, $\lambda = 1$ | -0.55 | -0.85 | -0.85 | -0.80 |
| advantage, $\lambda = 0.95$ | -0.43 | -0.77 | -0.81 | -0.80 |

**Reading the table**

- the **TD error** $\delta_\tt = \rew_\tt + \gamma\,\statevalfun_\phi(\text{next prefix}) - \statevalfun_\phi(\text{prefix})$ (B3)
    - token 1 has $\delta = +0.30$: the critic liked the prefix after it more than before
    - token 4 has $\delta = -0.80$: the score was far below what the critic expected at that prefix
- $\lambda = 0$ gives $\delta$ itself: token 1 is **praised**
- $\lambda = 1$ gives the total reward minus the critic's baseline: every token is told about the blunder
    - token 1 gets $-0.55$, the same sign as the whole completion
    - tokens 2 and 3 get $-0.85$, more blame than token 4
        - the critic valued their prefixes at $0.9$
        - what followed earned only $0.05$
- $\lambda = 0.95$ is between
    - the critic's estimate of the early prefix is trusted a little, the long tail of random outcomes a little less (B3 appendix)
- **no critic, no choice:** GRPO gives all four tokens the same number (C2)

<img src="images/rllm5_gae.png" width=75%>

- the same four tokens under three settings of $\lambda$, from the table
- the example is short; real completions have hundreds of tokens


## What the critic costs on a language model

**Why this section?** The critic's credit assignment is not free: it is the cost that motivates GRPO.

> **The critic is a second network of the policy's size, trained from a start where its values are meaningless, and it must keep up with a policy that keeps moving.**

- **memory and compute:** the critic has its own parameters, gradients and optimizer state, on top of the policy's
- **a value head on a shared backbone** is common
    - one network, two outputs: next-token distribution and a value
    - saves memory, but the policy and critic gradients compete for the same weights
    - the alternative is a separate critic network: no competition, about twice the memory
- **the cold start:** the policy begins as a good model
    - the critic's value head begins random
    - early in training the critic's errors make the advantages noisy
    - some recipes first train only the critic, or start it from a reward model

**A critic that falls behind**

- the policy changes, and the critic's targets change with it
- a lagging critic gives advantages that are biased or noisy, and the policy then follows noise
- two reasons it is hard on long completions
    - one score at the end must be spread over hundreds of tokens
    - the value of an early prefix is the average of many possible continuations: hard to learn from few samples
- recent work on long reasoning chains revisits exactly this
- RLLM-6 shows the alternative: no critic at all
- libraries package the heavy part (the value head, the value loss) and change their interfaces between releases: check the version you use


## Reading a PPO run: what to watch

**Why this section?** A run fails quietly. These signals tell you which part is failing.

> **Reward alone is not enough: a rising reward with a growing KL can mean the proxy is being exploited.**

| Signal | What it measures | A warning looks like |
|:--|:--|:--|
| mean reward (or score) | what the policy is optimizing | rises while real quality does not (the proxy is exploited) |
| KL to the reference | how far the policy has moved | grows without bound; or stays at $0$ (nothing is learned) |
| value loss | the critic's fit | high and not falling: advantages are noise |
| clip fraction | share of tokens whose ratio hit the clip | very high: steps too large; very low: steps too timid (always $0$ with one update per batch) |
| entropy of the policy | how spread the next-token distribution is | collapses early: exploration is gone |
| completion length | what is being produced | grows suspiciously: the reward may favor length |

**Reward hacking, once more**

- the reward is a proxy (C1), and optimization finds its weak spots
    - for a learned reward model: odd phrasing, length, flattery
    - for a rule: whatever the rule fails to forbid
- the **KL reward** limits the damage, and the **signals above** show it happening
- a larger $\beta$ is a trade, not a cure: it also slows the real improvement
- reward-model over-optimization (RLLM-2) has the same shape: the true quality peaks, then falls, while the proxy keeps rising


## PPO and GRPO, side by side

**Why this section?** This is the trade at the heart of C2, now with the details of this session.

> **PPO buys per-token credit with a critic. GRPO gives up per-token credit to remove it.**

| | PPO | GRPO |
|:--|:--|:--|
| **baseline** | learned: the critic's value of each prefix | computed: the mean reward of the group |
| **advantage** | per token (GAE) | one per completion, shared by its tokens |
| **networks in training** | policy and critic | policy only |
| **KL term** | a reward at every token | a loss term (RLLM-6) |
| **needs** | a critic that tracks the policy | enough completions per prompt, with varied rewards |
| **credit** | can tell a good token from a bad one in the same completion | cannot; a good completion lifts all its tokens |

**When each is natural**

- PPO fits when per-token credit matters and a critic can be trained: long dialogues, dense or shaped rewards, a reward model with a fine scale
- GRPO fits when a whole completion is judged and a group is cheap: verifiable rewards, short feedback loops
- neither is free of cost
    - GRPO spends compute on many completions per prompt instead of on a critic
    - PPO spends it on the critic
- the weak spots of GRPO (C2, "known weak spots") are the subject of RLLM-6


# What we established, and what it sets up

- **Established**
    - four networks: the policy and the critic learn, the reference and the reward model are frozen
    - the old policy is a snapshot
    - one iteration: roll out, add the score and the per-token KL reward, compute advantages with the critic and GAE, update for $K$ epochs, refresh
    - the critic turns one score into an advantage per token
    - $\lambda$ sets how far each token feels the end
    - the critic's cost: a second large network, a cold start, and the need to keep up with the policy
    - the run signals: reward, KL, value loss, clip fraction, entropy, length
    - PPO against GRPO: per-token credit against no critic
- **Sets up**
    - **the limitation:** a critic is costly and fragile on long completions, and GRPO's shared advantage is coarse
    - next session, RLLM-6: GRPO and its refinements
